# 🧬 JORINOVA NEXUS — Histopathology (tissue) CLASSIFIER (fine-tuning)

Classifies a tissue patch as benign / malignant / tumour type. This is a **classification**
model (`yolov8*-cls`) — the right task for patch-labelled histopathology sets, reaching far
higher accuracy than forcing detection boxes (metric = **top-1 accuracy**, target ~0.95).

Produces `histology.pt`; the predicted class resolves to its finding via
`backend/ai_services/histology_findings.json`.

> **Fine-tuning, NOT from scratch** (starts from `yolov8m-cls.pt`).
> **Keyless of Kaggle** — data from Roboflow (your free key), no GitHub token.

**Before you start:** Runtime -> **Change runtime type** -> **T4 GPU**.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > T4 GPU, then rerun'

## 2. Install training dependencies

In [ ]:
%pip -q install ultralytics roboflow
import ultralytics, torch
print('ultralytics', ultralytics.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 3. Mount Drive (checkpoints survive a runtime reset)

In [ ]:
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/nexus_histology', exist_ok=True)
    print('Drive ready: /content/drive/MyDrive/nexus_histology')
except Exception as e:
    print('Drive not mounted (optional):', e)

## 4. Get the training data (Roboflow — no Kaggle, no GitHub token)

In [ ]:
# ── 4. Classification data (Roboflow) -> train/val class folders (DATA_DIR) ──
# Works whether the Roboflow project is a CLASSIFICATION ('folder') or DETECTION
# ('yolov8') export: detection boxes are cropped into per-class images.
# Add your own from https://universe.roboflow.com/search?q=histopathology  (or histology).
import os, glob, yaml, shutil, random
from getpass import getpass
from PIL import Image
key = getpass('Roboflow Private API Key: ').strip(); assert key, 'EMPTY key — paste the Private API Key.'
os.environ['ROBOFLOW_API_KEY'] = key
try:
    from roboflow import Roboflow
except ModuleNotFoundError:
    import subprocess, sys; subprocess.run([sys.executable,'-m','pip','install','-q','roboflow'], check=True)
    from roboflow import Roboflow
rf = Roboflow(api_key=key)

CANDIDATES = [
    ('yolo-usyrb', 'lung-cancer-qmdnh-kps7i'),
]
RAW, CLS = '/content/data/histo_raw', '/content/data/histo_cls'
for d in (RAW, CLS):
    if os.path.exists(d): shutil.rmtree(d)

ds = fmt = None
for w, p in CANDIDATES:
    try:
        proj = rf.workspace(w).project(p)
        vers = sorted([v.version for v in proj.versions()], reverse=True) or list(range(11, 0, -1))
        for v in vers:
            for f in ('folder', 'yolov8'):        # prefer classification 'folder', fall back to detection
                try:
                    ds = proj.version(v).download(f, location=RAW); fmt = f
                    print('OK', w+'/'+p, 'v'+str(v), 'as', f); break
                except Exception as e:
                    print('  v%s %s -> %r' % (v, f, str(e)[:90]))
            if ds: break
        if ds: break
    except Exception as e:
        print('skip %s/%s -> %r' % (w, p, str(e)[:90]))
assert ds, 'None downloaded — read errors above (401/403=key, 404=slug). Add a project from the search URL.'

if fmt == 'folder':                                # already class subfolders (train/valid/test/<class>/*)
    for src in ('train', 'valid', 'val', 'test'):
        base = os.path.join(ds.location, src)
        if not os.path.isdir(base): continue
        split = 'train' if src == 'train' else 'val'
        for cls in os.listdir(base):
            cdir = os.path.join(base, cls)
            if os.path.isdir(cdir):
                dst = os.path.join(CLS, split, cls); os.makedirs(dst, exist_ok=True)
                for ip in glob.glob(cdir + '/*.*'): shutil.copy(ip, dst)
else:                                              # detection export -> crop each box into a class folder
    dy = yaml.safe_load(open(os.path.join(ds.location, 'data.yaml'))); nm = dy['names']
    nm = [nm[k] for k in sorted(nm)] if isinstance(nm, dict) else list(nm)
    for src in ('train', 'valid', 'test'):
        idir = os.path.join(ds.location, src, 'images'); ldir = os.path.join(ds.location, src, 'labels')
        if not os.path.isdir(idir): continue
        split = 'train' if src == 'train' else 'val'
        for ip in glob.glob(idir + '/*.*'):
            lp = os.path.join(ldir, os.path.splitext(os.path.basename(ip))[0] + '.txt')
            if not os.path.exists(lp): continue
            try: im = Image.open(ip).convert('RGB')
            except Exception: continue
            W, H = im.size
            for j, ln in enumerate(open(lp)):
                q = ln.split()
                if len(q) < 5: continue
                c = int(float(q[0])); xc, yc, bw, bh = map(float, q[1:5])
                x1, y1 = max(0, int((xc-bw/2)*W)), max(0, int((yc-bh/2)*H))
                x2, y2 = min(W, int((xc+bw/2)*W)), min(H, int((yc+bh/2)*H))
                if x2-x1 < 8 or y2-y1 < 8: continue
                cls = nm[c] if c < len(nm) else str(c)
                out = os.path.join(CLS, split, cls); os.makedirs(out, exist_ok=True)
                im.crop((x1, y1, x2, y2)).save(os.path.join(out, '%s_%d.jpg' % (os.path.splitext(os.path.basename(ip))[0], j)))

# classification needs train/ + val/ — carve a val split if the export had none
if not os.path.isdir(os.path.join(CLS, 'val')) or not os.listdir(os.path.join(CLS, 'val')):
    for cls in os.listdir(os.path.join(CLS, 'train')):
        files = glob.glob(os.path.join(CLS, 'train', cls) + '/*'); random.shuffle(files)
        dst = os.path.join(CLS, 'val', cls); os.makedirs(dst, exist_ok=True)
        for fp in files[:max(1, len(files)//6)]: shutil.move(fp, dst)

DATA_DIR = CLS
counts = {s: {c: len(os.listdir(os.path.join(CLS, s, c))) for c in sorted(os.listdir(os.path.join(CLS, s)))}
          for s in ('train', 'val') if os.path.isdir(os.path.join(CLS, s))}
print('DATA_DIR =', DATA_DIR)
print('classes:', sorted(os.listdir(os.path.join(CLS, 'train'))))
print('counts:', counts)
print('NOTE: if these class names differ from histology_findings.json keys, send me the list and I add aka-aliases.')

## 5. Choose base weights (fine-tune, never from scratch)

In [ ]:
BASE_WEIGHTS = 'yolov8m-cls.pt'   # classification model (yolov8s-cls = faster/less accurate)
print('fine-tuning (classification) from:', BASE_WEIGHTS)

## 6. Fine-tune the classifier
Metric is **top-1 accuracy** (not mAP). Tissue patches → `imgsz=384` (texture detail),
`yolov8m-cls`, ~120 epochs, cosine LR + light dropout/erasing. Drop `batch` to 16 if you hit
CUDA OOM. H&E stain varies between labs — HSV augmentation helps generalisation.

In [ ]:
from ultralytics import YOLO
import os
PROJECT = '/content/drive/MyDrive/nexus_histology/runs'
try: os.makedirs(PROJECT, exist_ok=True)
except Exception: PROJECT = '/content/runs/classify'
model = YOLO(BASE_WEIGHTS)   # yolov8m-cls -> classification (patch-labelled histopathology)
model.train(data=DATA_DIR, epochs=120, imgsz=384, batch=32,
            project=PROJECT, name='histology', pretrained=True, patience=30, plots=True,
            cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180,
            hsv_h=0.02, hsv_s=0.7, hsv_v=0.4, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 6b. RESUME if a run disconnects (run INSTEAD of step 6)

In [ ]:
import os, glob
from ultralytics import YOLO
runs = sorted(glob.glob('/content/drive/MyDrive/nexus_histology/runs/**/weights/last.pt', recursive=True), key=os.path.getmtime)
loose = [p for p in sorted(glob.glob('/content/drive/MyDrive/nexus_histology/**/*.pt', recursive=True)
                           + glob.glob('/content/*.pt'), key=os.path.getmtime)
         if 'yolov8' not in os.path.basename(p).lower()]
assert runs or loose, 'No checkpoint yet — run step 6 first.'
ckpt = (runs or loose)[-1]; print('checkpoint:', ckpt)
model = None
try:
    if ckpt in runs: model = YOLO(ckpt); model.train(resume=True)
    else: raise RuntimeError('loose')
except Exception as e:
    print('continue-from-weights:', str(e)[:80]); model = None
if model is None:
    assert 'DATA_DIR' in globals(), 'Re-run step 4 first so DATA_DIR is set.'
    model = YOLO(ckpt); model.train(data=DATA_DIR, epochs=120, imgsz=384, batch=32,
        project='/content/drive/MyDrive/nexus_histology/runs', name='histology_cont', pretrained=True, patience=30,
        cos_lr=True, dropout=0.2, fliplr=0.5, flipud=0.5, degrees=180, erasing=0.2)
m = model.val(); print(f'top1 = {m.top1:.3f}   top5 = {m.top5:.3f}')

## 7. Export the model (Drive backup + download)

In [ ]:
import shutil, glob, os
DRIVE = '/content/drive/MyDrive/nexus_histology'
best = sorted([c for c in glob.glob('/content/**/weights/best.pt', recursive=True)
               + glob.glob(DRIVE + '/**/weights/best.pt', recursive=True) if os.path.isfile(c)],
              key=os.path.getmtime)[-1]
print('using weights:', best)
from ultralytics import YOLO; print('classes:', YOLO(best).model.names)
try: os.makedirs(DRIVE, exist_ok=True); shutil.copy(best, DRIVE + '/histology.pt'); print('Drive ->', DRIVE + '/histology.pt')
except Exception as e: print('drive copy skipped:', e)
from google.colab import files; files.download(best)

## 8. Put it in the app
1. Rename the download to **`histology.pt`** at **`backend/models/histology/histology.pt`**, commit, push.
2. Auto-loads for `image_type == "histology"`; the predicted class resolves via `backend/ai_services/histology_findings.json`. The vision service **detects classification models automatically** (reads top-1 + top-k, no code change).
3. On Render, build with `INSTALL_ML=1` (>=2 GB); else Claude vision reads the field.

> ⚠️ Screening aid only — a pathologist grades/stages and issues the diagnosis. If your dataset's class names differ from the map keys, send me the printed `classes:` list and I'll add aka-aliases.